# Part 2: How Real VLAs Represent Actions

## Notebook 7 — π₀.₅ (pi0.5): Flow Matching + adaRMS

pi0.5 is Physical Intelligence's generalization-focused successor to pi0. In the leRobot implementation, it uses **flow matching with adaRMS conditioning** — an improved version of pi0's architecture. Notably, it does NOT use FAST tokenization in leRobot.


### 1. Load pi0.5 configuration

Compare with pi0 — the config is very similar but with adaRMS conditioning for better generalization across heterogeneous data.


In [ ]:
from lerobot.policies.pi05.configuration_pi05 import PI05Config

cfg = PI05Config()
print(f"Policy type: pi0.5 (Flow Matching VLA + adaRMS)")
print(f"Action expert:       {cfg.action_expert_variant}")  # gemma_300m
print(f"Chunk size:          {cfg.chunk_size}")  # 50
print(f"Action steps:        {cfg.n_action_steps}")  # 50
print(f"Max action dim:      {cfg.max_action_dim}")  # 32
print(f"Train expert only:   {cfg.train_expert_only}")  # False


### 2. pi0 vs pi0.5: what changed?

pi0.5 adds **adaRMS (Adaptive Root Mean Square)** conditioning. This is a normalization technique that helps the model handle heterogeneous data sources (different robots, cameras, environments).

Critically: in leRobot v0.6.0, pi0.5 does NOT include FAST tokenization. It's a pure flow matching model with an action expert.


In [ ]:
# Side-by-side config comparison
print("pi0 config keys with action/token relevance:")
print("  action_expert_variant: gemma_300m")
print("  chunk_size: 50")
print("  n_action_steps: 50")
print("  max_action_dim: 32")
print("  train_expert_only: False")

print("\npi0-FAST config keys (for contrast):")
print("  action_tokenizer_name: lerobot/fast-action-tokenizer")
print("  max_action_tokens: 256")
print("  fast_skip_tokens: 128")
print("  validate_action_token_prefix: True")

print("\npi05 does NOT have action_tokenizer_name or max_action_tokens.")
print("→ pi0.5 uses flow matching, NOT FAST tokenization (in leRobot).")


### 3. adaRMS: adaptive conditioning for generalization

adaRMS normalizes activations based on RMS statistics, conditioned on which data source (robot embodiment) the sample comes from. This lets the model handle diverse robot morphologies without them interfering.


In [ ]:
# adaRMS in pi0.5:
# adarms_cond_dim is set in the Gemma expert config
# It conditions each layer's normalization on the embodiment identity

print("adaRMS Conditioning:")
print("  Problem: Different robots have different action scales")
print("    - Franka: joints in radians, range ~[-π, π]")
print("    - UR5: joints in radians, different range")
print("    - Mobile base: position deltas in meters")

print("  Solution: adaRMS conditions each layer on robot identity")
print("    - Learns per-embodiment scaling/shifting parameters")
print("    - Shared model weights + per-robot conditioning")
print("    → One model handles many robots without conflict")


### 4. Open-world generalization

pi0.5's main claim: performs manipulation tasks in UNSEEN environments. The training mixture spans 10+ robot embodiments + web vision-language data. Co-training (not sequential fine-tuning) preserves general knowledge.


In [ ]:
# pi0.5 Training Data (approximate, from PI blog)
print("pi0.5 Training Data:")
print("  Robot data: 10+ embodiments")
print("    - Franka Panda, UR5, ALOHA bi-manual")
print("    - Mobile manipulators, humanoid upper-body")
print("  Web data: vision-language (image-caption, VQA)")
print("  Training: CO-TRAINING (mixed, not sequential)")

print("Key claim: Co-training preserves web knowledge")
print("  while adding robot control capability.")
print("  Sequential fine-tuning tends to forget general knowledge.")


### 5. Where does pi0.5 fit in the action tokenization story?

Physical Intelligence released pi0-FAST (autoregressive FAST tokens) alongside pi0 (flow matching). Then they released pi0.5, which in leRobot's implementation goes back to **pure flow matching with an action expert** — no FAST tokens.

This tells us: even the creators of FAST recognized that continuous flow matching with a dedicated action expert is still the superior approach for dexterous control. Tokenization gives training speed but loses action precision.


In [ ]:
# The pendulum swing
print("Physical Intelligence Action Representation Evolution:")
print("  pi0 (2024):      Flow matching + action expert")
print("  pi0-FAST (2025): FAST tokens (autoregressive, 5× faster training)")
print("  pi0.5 (2025-26): BACK to flow matching + action expert + adaRMS")

print("\nWhy go back? Possible reasons:")
print("  1. Autoregressive inference is SLOW (many tokens per chunk)")
print("  2. Tokenization quantization error loses fine dexterity")
print("  3. Flow matching produces smoother trajectories")
print("  4. Action expert specializes better than shared backbone")


### Where This Leaves Us

pi0.5 completes the arc: from continuous (pi0) → tokenized (pi0-FAST) → back to continuous (pi0.5). The action expert persists throughout. Tokenization was tried but the field continues to favor continuous action representations.
